# Laboratorio 09: Bag of Words, Similitud de Coseno e Índice Invertido

## Introducción

Pipeline completo de recuperación de información textual. Para búsqueda booleana y por similitud se implementan
**dos versiones**: enfoque **lineal** (línea base) y enfoque **optimizado** con índice invertido. La parte central es el
análisis experimental comparativo.

## Parte 1: Configuración y Carga de Datos

```sql
CREATE TABLE noticias (
    id SERIAL PRIMARY KEY,
    url TEXT,
    contenido TEXT,
    categoria VARCHAR(50),
    bag_of_words JSONB
);

CREATE TABLE stopwords (
    id SERIAL PRIMARY KEY,
    word TEXT UNIQUE NOT NULL
);
```

Cargar el dataset `news_es.csv` en `noticias` y `stoplist_es.txt` en `stopwords`.

> Las tablas se crean y se cargan automáticamente al hacer `docker compose up` (ver `init.sql`).

In [1]:
import psycopg2, pandas as pd, json, time, math, os, re
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from psycopg2.extras import execute_batch
from nltk.stem.snowball import SnowballStemmer

# pandas avisa al recibir una conexión psycopg2 en read_sql; funciona igual, solo silenciamos el aviso
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

def connect_db():
    conn = psycopg2.connect(
        dbname="labofwords", user="postgres",
        password="postgres", host="localhost", port=os.getenv("PGPORT", "5432")
    )
    return conn

In [2]:
def fetch_data():
    conn = connect_db()
    df = pd.read_sql("SELECT id, contenido FROM noticias;", conn)
    conn.close()
    return df

noticias_df = fetch_data()

## Parte 2 (3 puntos): Preprocesamiento y Bag of Words

Pipeline de preprocesamiento (Document → Tokenization → Stop words removal → Stemming):

1. **Normalización**: todo el texto se pasa a minúsculas.
2. **Tokenización**: se extraen solo secuencias de letras (`[a-záéíóúüñ]+`), descartando números y signos de puntuación.
3. **Stopwords**: se eliminan las palabras de la tabla `stopwords` (cargada desde `stoplist.txt`) y los tokens de una sola letra.
4. **Stemming**: `SnowballStemmer("spanish")` de NLTK reduce cada palabra a su raíz (*tecnologías → tecnolog*, *inflación → inflacion*), para que coincidan palabras de la misma familia léxica.

El **Bag of Words** de cada noticia es el conteo de cada raíz (`tf`), y se guarda como `JSONB` en la columna `bag_of_words`.

In [3]:
def load_stopwords():
    conn = connect_db()
    with conn.cursor() as cur:
        cur.execute("SELECT word FROM stopwords;")
        words = {row[0].strip().lower() for row in cur.fetchall()}
    conn.close()
    return words

STOPWORDS = load_stopwords()
stemmer = SnowballStemmer("spanish")
TOKEN_RE = re.compile(r"[a-záéíóúüñ]+")

def preprocess(text):
    if not text:
        return []
    tokens = TOKEN_RE.findall(text.lower())                                  # minúsculas + tokenización (solo letras)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]        # stopwords
    return [stemmer.stem(t) for t in tokens]                                 # stemming

def compute_bow(text):
    return dict(Counter(preprocess(text)))

def update_bow_in_db(dataframe):
    rows = [(json.dumps(compute_bow(text), ensure_ascii=False), int(doc_id))
            for doc_id, text in zip(dataframe["id"], dataframe["contenido"])]
    conn = connect_db()
    with conn.cursor() as cur:
        execute_batch(cur, "UPDATE noticias SET bag_of_words = %s::jsonb WHERE id = %s;", rows, page_size=500)
    conn.commit()
    conn.close()
    print(f"BoW actualizado para {len(rows)} documentos")

def fetch_data_bow():
    conn = connect_db()
    df = pd.read_sql("SELECT id, contenido, bag_of_words FROM noticias ORDER BY id;", conn)
    conn.close()
    display(df.head(10))
    return df

update_bow_in_db(noticias_df)
noticias_bow_df = fetch_data_bow()

BoW actualizado para 1217 documentos


,id,contenido,bag_of_words
0,1,Durante el foro La banca articulador empresari...,"{'aca': 1, 'adn': 1, 'cas': 1, 'for': 1, 'ret'..."
1,2,El regulador de valores de China dijo el domin...,"{'dat': 1, 'deb': 1, 'inc': 1, 'med': 1, 'mes'..."
2,3,En una industria históricamente masculina como...,"{'go': 1, 'us': 3, 'air': 1, 'baj': 1, 'ceo': ..."
3,4,Con el dato de marzo el IPC interanual encaden...,"{'agu': 1, 'car': 2, 'cas': 1, 'dat': 2, 'ine'..."
4,5,Ayer en Cartagena se dio inicio a la versión n...,"{'agu': 1, 'cup': 2, 'deb': 2, 'etc': 1, 'for'..."
5,6,El emprendimiento del sector inmobiliario Rent...,"{'us': 2, 'big': 1, 'ceo': 1, 'dat': 1, 'sum':..."
6,7,El país tiene pendiente generar más y mejores ...,"{'ve': 1, 'baj': 1, 'cas': 2, 'dat': 1, 'dej':..."
7,8,En el 'podcast' tenemos el placer de entrevist...,"{'agil': 4, 'habl': 1, 'mund': 1, 'nach': 3, '..."
8,9,"El factor humano, normalmente identificado com...","{'ali': 2, 'bas': 1, 'cas': 2, 'cin': 1, 'cuy'..."
9,10,Los perfiles que se demandan en un banco han e...,"{'are': 1, 'cab': 1, 'cas': 1, 'dat': 2, 'pud'..."


## Parte 3 (3 puntos): Búsqueda Booleana, Lineal vs. Índice Invertido

### 3.1 Búsqueda Lineal

Recorre todos los documentos verificando si cada BoW satisface la consulta.

- La consulta tiene la forma `t1 OP t2 OP t3 ...` con `OP ∈ {AND, OR, AND-NOT}` y se evalúa de izquierda a derecha.
- Cada término de la consulta pasa por el **mismo preprocesamiento** que los documentos (`economía → econom`); si no, nunca coincidiría con las raíces guardadas en el BoW.
- Complejidad: **O(N)** evaluaciones (una por documento), independientemente de cuántos documentos contengan los términos.

In [4]:
OPERATORS = {"AND", "OR", "AND-NOT"}

def normalize_term(word):
    tokens = preprocess(word)
    return tokens[0] if tokens else stemmer.stem(word.lower())

def parse_boolean_query(query):
    parts = query.split()
    terms = [normalize_term(p) for p in parts[0::2]]
    ops = [p.upper() for p in parts[1::2]]
    if len(terms) != len(ops) + 1 or any(op not in OPERATORS for op in ops):
        raise ValueError(f"Consulta booleana inválida: {query!r}")
    return terms, ops

def eval_boolean(terms, ops, contains):
    result = contains(terms[0])
    for op, term in zip(ops, terms[1:]):
        if op == "AND":
            result = result and contains(term)
        elif op == "OR":
            result = result or contains(term)
        else:  
            result = result and not contains(term)
    return result

def boolean_search_linear(query, dataframe, verbose=True):
    terms, ops = parse_boolean_query(query)
    result = [doc_id for doc_id, bow in zip(dataframe["id"], dataframe["bag_of_words"])
              if eval_boolean(terms, ops, lambda t: t in bow)]
    if verbose:
        print(f"{query!r} -> términos {terms}: {len(result)} documentos, primeros ids: {result[:10]}")
    return result

boolean_search_linear("economía AND inflación",       noticias_bow_df)
boolean_search_linear("banca OR finanzas",            noticias_bow_df)
boolean_search_linear("tecnología AND-NOT inflación", noticias_bow_df);

'economía AND inflación' -> términos ['econom', 'inflacion']: 244 documentos, primeros ids: [5, 7, 18, 21, 24, 32, 33, 34, 37, 39]
'banca OR finanzas' -> términos ['banc', 'finanz']: 547 documentos, primeros ids: [1, 2, 5, 9, 10, 12, 13, 15, 16, 17]
'tecnología AND-NOT inflación' -> términos ['tecnolog', 'inflacion']: 306 documentos, primeros ids: [12, 13, 14, 15, 16, 17, 19, 20, 23, 28]


### 3.2 Búsqueda con Índice Invertido

- **Índice**: para cada término `t` se guarda su *posting list* `[(doc_id, tf), ...]` **ordenada por `doc_id`**, requisito para poder mezclar listas en tiempo lineal.
- **IDF**: `idf_t = log10(N / df_t)`, donde `df_t` es la longitud de la posting list de `t`.
- **Longitud (norma)** de cada documento: `length_d = sqrt(Σ_t w_{t,d}^2)` con `w_{t,d} = log10(1 + tf_{t,d}) × idf_t`. Se precalcula aquí para que la Parte 4 normalice el coseno sin recorrer el documento completo.
- **AND / OR / AND-NOT**: mezcla con dos punteros sobre listas ordenadas, en **O(n + m)**:
  - `AND`: avanza el puntero con el `doc_id` menor y agrega cuando coinciden (intersección).
  - `OR`: igual, pero además agrega los `doc_id` que no coinciden y el resto de ambas listas (unión).
  - `AND_NOT`: agrega los `doc_id` de la primera lista que no aparecen en la segunda (diferencia).
- `L(word)` aplica el mismo preprocesamiento antes de buscar el término en el diccionario.
- `boolean_search_index(query, idx)` resuelve la misma consulta textual que la versión lineal usando el índice; se usa para verificar que ambos enfoques retornan exactamente los mismos documentos y en el experimento de la Parte 5.

In [5]:
"""
indice = {
    "word1": [("doc1", tf1), ("doc2", tf2), ("doc3", tf3)],
    "word2": [("doc2", tf2), ("doc4", tf4)],
    "word3": [("doc3", tf3), ("doc5", tf5)],
}
idf = {
    "word1": 3,
    "word2": 2,
    "word3": 2,
}
length = {
    "doc1": 15.5236,
    "doc2": 10.5236,
    "doc3": 5.5236,
}
"""
class InvertedIndex:
    def __init__(self):
        self.index  = {}
        self.idf    = {}
        self.length = {}
        self._docs  = {}
        self.N      = 0

    def build_from_db(self, doc_ids=None):
        conn = connect_db()
        with conn.cursor() as cur:
            if doc_ids is None:
                cur.execute("SELECT id, contenido, bag_of_words FROM noticias ORDER BY id;")
            else:
                cur.execute("SELECT id, contenido, bag_of_words FROM noticias WHERE id = ANY(%s) ORDER BY id;",
                            (list(map(int, doc_ids)),))
            rows = cur.fetchall()
        conn.close()
        self.build(rows)

    def build(self, rows):
        self.index, self.idf, self.length, self._docs = {}, {}, {}, {}
        for doc_id, contenido, bow in rows:
            self._docs[doc_id] = contenido
            for term, tf in bow.items():
                self.index.setdefault(term, []).append((doc_id, tf))
        self.N = len(self._docs)

        for term, postings in self.index.items():
            self.idf[term] = math.log10(self.N / len(postings))

        sq = dict.fromkeys(self._docs, 0.0)
        for term, postings in self.index.items():
            idf = self.idf[term]
            for doc_id, tf in postings:
                sq[doc_id] += (math.log10(1 + tf) * idf) ** 2
        self.length = {doc_id: math.sqrt(s) for doc_id, s in sq.items()}

    def L(self, word):
        return self.index.get(normalize_term(word), [])

    def showDocument(self, doc_id, max_chars=80):
        return self._docs.get(doc_id, "")[:max_chars] + "..."

    def showDocuments(self, posting_list, max_chars=80):
        return [self.showDocument(doc_id, max_chars) for doc_id, *_ in posting_list]

    def cosine_search_linear(self, query, top_k=5):
        pass

    def cosine_search(self, query, top_k=5):
        score = {}
        return sorted(score.items(), key=lambda t: t[1], reverse=True)[:top_k]

def AND(list1, list2):
    answer, i, j = [], 0, 0
    while i < len(list1) and j < len(list2):
        d1, d2 = list1[i][0], list2[j][0]
        if d1 == d2:
            answer.append(list1[i])
            i += 1; j += 1
        elif d1 < d2:
            i += 1
        else:
            j += 1
    return answer

def OR(list1, list2):
    answer, i, j = [], 0, 0
    while i < len(list1) and j < len(list2):
        d1, d2 = list1[i][0], list2[j][0]
        if d1 == d2:
            answer.append(list1[i])
            i += 1; j += 1
        elif d1 < d2:
            answer.append(list1[i]); i += 1
        else:
            answer.append(list2[j]); j += 1
    answer.extend(list1[i:])
    answer.extend(list2[j:])
    return answer

def AND_NOT(list1, list2):
    answer, i, j = [], 0, 0
    while i < len(list1):
        if j >= len(list2) or list1[i][0] < list2[j][0]:
            answer.append(list1[i]); i += 1
        elif list1[i][0] == list2[j][0]:
            i += 1; j += 1
        else:
            j += 1
    return answer

def boolean_search_index(query, idx):
    terms, ops = parse_boolean_query(query)
    result = idx.index.get(terms[0], [])
    for op, term in zip(ops, terms[1:]):
        result = {"AND": AND, "OR": OR, "AND-NOT": AND_NOT}[op](result, idx.index.get(term, []))
    return [doc_id for doc_id, *_ in result]

idx = InvertedIndex()
idx.build_from_db()
print(f"Índice construido: {idx.N} documentos, {len(idx.index)} términos")

Índice construido: 1217 documentos, 13307 términos


**Pruebas (mínimo 5, combinando AND, OR, AND-NOT)**

In [6]:
def mostrar(titulo, posting_list, k=5):
    docs = idx.showDocuments(posting_list)
    print(f"{titulo}: {len(docs)} documentos")
    for d in docs[:k]:
        print("   -", d)

mostrar("sostenibilidad AND ambiente AND renovables",
        AND(idx.L("sostenibilidad"), AND(idx.L("ambiente"), idx.L("renovables"))))
mostrar("tecnología AND (banca OR finanzas)",
        AND(idx.L("tecnología"), OR(idx.L("banca"), idx.L("finanzas"))))
mostrar("economía AND-NOT inflación",
        AND_NOT(idx.L("economía"), idx.L("inflación")))
mostrar("(digital OR innovación) AND-NOT banca",
        AND_NOT(OR(idx.L("digital"), idx.L("innovación")), idx.L("banca")))
mostrar("energía AND (solar OR eólica)",
        AND(idx.L("energía"), OR(idx.L("solar"), idx.L("eólica"))))
mostrar("(alianza AND empresa) AND-NOT crisis",
        AND_NOT(AND(idx.L("alianza"), idx.L("empresa")), idx.L("crisis")))

for q in ["economía AND inflación", "banca OR finanzas", "tecnología AND-NOT inflación",
          "energía AND solar OR eólica", "digital OR innovación AND-NOT banca"]:
    lineal = boolean_search_linear(q, noticias_bow_df, verbose=False)
    indice = boolean_search_index(q, idx)
    print(f"{q!r}: lineal={len(lineal)} índice={len(indice)} iguales={lineal == indice}")

sostenibilidad AND ambiente AND renovables: 9 documentos
   - La importancia que cada día más empresas le otorgan al tema de la sostenibilidad...
   - Esta tipología de energía, clave para la sostenibilidad del planeta, nace de la ...
   - El turismo sostenible persigue desarrollar su actividad generando un impacto mín...
   - En un intento por aumentar el acceso al financiamiento climático, IFC otorgó un ...
   - Este jueves se conoció que IFC otorgó un préstamo de hasta US200 millones a Bbva...
tecnología AND (banca OR finanzas): 174 documentos
   - BBVA ha presentado BBVA Spark, su propuesta integral de servicios financieros pa...
   - Con acciones que se comprometen con el desarrollo sostenible, ambas organizacion...
   - El regulador bancario de China dijo que las recientes medidas para frenar el mon...
   - JPMorgan dijo el lunes que los cambios regulatorios chinos eran un problema loca...
   - Bancolombia y Prosegur Alarms se unen con el fin de ofrecerle a los usuarios de ...
ec

## Parte 4 (3 puntos): Similitud de Coseno, Lineal vs. Índice Invertido

### 4.1 Coseno Lineal

Calcula TF-IDF y similitud de coseno contra **todos** los documentos sin usar el índice.

In [7]:
def cosine_search_linear(self, query, top_k=5):
    pass

InvertedIndex.cosine_search_linear = cosine_search_linear

### 4.2 Coseno con Índice Invertido

Solo evalúa documentos candidatos que comparten al menos un término con la query.

In [8]:
def cosine_search(self, query, top_k=5):
    score = {}
    # TODO
    return sorted(score.items(), key=lambda t: t[1], reverse=True)[:top_k]

InvertedIndex.cosine_search = cosine_search

**Pruebas: verificar que ambos retornan los mismos documentos**

In [9]:
queries = [
    "innovaciones en banca digital y tecnología financiera",
    "inflación y crecimiento económico",
    "sostenibilidad y energías renovables"
]
for q in queries:
    print(f"\n{q}")
    print("Lineal :", idx.cosine_search_linear(q, 5))
    print("Índice :", idx.cosine_search(q, 5))


innovaciones en banca digital y tecnología financiera
Lineal : None
Índice : []

inflación y crecimiento económico
Lineal : None
Índice : []

sostenibilidad y energías renovables
Lineal : None
Índice : []


## Parte 5 (3 puntos): Experimento, Medición de Tiempos

Diseñar e implementar un experimento que mida el tiempo promedio de ejecución de los cuatro enfoques (booleana
lineal, booleana con índice, coseno lineal, coseno con índice) variando el tamaño de la colección.

Generar **dos gráficos** (uno para búsqueda booleana y uno para coseno), cada uno con:

- **Eje X**: número de documentos.
- **Eje Y**: tiempo de ejecución promedio (segundos).
- Dos curvas: lineal vs. índice invertido.
- Título, etiquetas de ejes, leyenda y valores sobre los puntos.

## Parte 6 (8 puntos): Análisis de Resultados

Con base en los gráficos obtenidos, redactar un análisis (mínimo 300 palabras) que responda las siguientes
preguntas. Cada respuesta debe sustentarse con datos concretos del experimento.

1. **(Complejidad)** ¿Los tiempos medidos son consistentes con la complejidad teórica de cada enfoque? Indicar la
   complejidad de los cuatro algoritmos y contrastarla con los resultados.
2. **(Punto de quiebre)** ¿A partir de qué tamaño de colección el índice invertido supera al enfoque lineal en cada
   tipo de búsqueda? ¿Coinciden ambos puntos de quiebre?
3. **(Overhead)** ¿Se observa algún costo adicional del índice para colecciones pequeñas? ¿A qué se debe?
4. **(Exactitud)** ¿Los resultados retornados por el enfoque lineal y el de índice son idénticos? Si hay diferencias
   en los scores, explique la causa.
5. **(Limitaciones)** ¿Qué problemas tendría el índice en memoria si la colección fuera 100× más grande? ¿Qué
   alternativa de almacenamiento existe?